# Week 6 - Rainbow DQN and BBF - Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Distinguish a distribution from a confidence interval. Start with **Exercises 3 and 6** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-distinguish-a-distribution-from-a-confidence-interval).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, torch, matplotlib.pyplot as plt
import _pkg_loader  # registers w06_project
from w06_project import agent as agent_mod, train as train_mod, solutions
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## 1. Double DQN target (two stars)
Compute $y = r + (1-d)\gamma\; Q_{\theta^-}(s', \arg\max_{a'} Q_\theta(s', a'))$.

Online net picks the argmax (carries selection noise); target net evaluates the chosen action. Their errors are correlated through shared training, so Double DQN reduces coupling without guaranteeing zero bias.

In [ ]:
def double_dqn_target(reward, done, q_on_next, q_tg_next, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
r = torch.tensor([1.0]); d = torch.tensor([0.0])
q_on = torch.tensor([[5., 1.]]); q_tg = torch.tensor([[3., 9.]])
out = select_exercise(solutions.exercise_1_double_dqn_target, double_dqn_target)(r, d, q_on, q_tg, 0.5)
assert torch.allclose(out, torch.tensor([1.0 + 0.5 * 3.0]))
print('exercise 1 OK')

## 2. Dueling head (two stars)
$Q(s,a) = V(s) + (A(s,a) - \bar A(s))$. Mean subtraction makes the effective advantage residual sum to zero and $V$ equal the action-uniform mean of Q. The raw advantage head can still be shifted by a constant without changing Q.

In [ ]:
def dueling_head(v, a):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
v = torch.tensor([[1.0]]); a = torch.tensor([[2., 4., 6.]])  # mean = 4
out = select_exercise(solutions.exercise_2_dueling_head, dueling_head)(v, a)
assert torch.allclose(out, torch.tensor([[-1., 1., 3.]]))
print('exercise 2 OK')

## 3. PER importance-sampling weight (two stars)
$w_i = (N \cdot P(i))^{-\beta}$, normalised by $\max_j w_j$ so $w_i \in [0, 1]$.

> **Hint**: $\beta=1$ gives exact correction before normalisation. A full-buffer maximum rescales the expectation; a minibatch maximum can add bias. common schedule is $\beta: 0.4 \to 1$ linearly over training.

In [ ]:
def per_weight(prob, N, beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = np.array([0.5, 0.3, 0.2])
w = select_exercise(solutions.exercise_3_per_weight, per_weight)(p, N=10, beta=0.4)
assert w.max() == 1.0 and (w >= 0).all()
print('exercise 3 OK')

## 4. PER priority update (one star)
After a training step, the new priorities at the sampled indices are $|\delta_i| + \epsilon$.

In [ ]:
def priority_update(old_priorities, idx, td_errors, eps):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = np.ones(5)
out = select_exercise(solutions.exercise_4_priority_update, priority_update)(p, np.array([1, 3]), np.array([-2.0, 0.5]), eps=1e-6)
assert out[1] > 1.999
print('exercise 4 OK')
np.testing.assert_allclose(out, [1., 2.000001, 1., .500001, 1.])


## 5. NoisyNet $f$ function (one star)
$f(x) = \text{sign}(x)\sqrt{|x|}$ - the factorised-Gaussian transform that lets $\epsilon^W_{ij} = f(\epsilon^{out}_i) f(\epsilon^{in}_j)$ share input/output factors. Those weight-noise entries are dependent, unlike independent Gaussian noise.

In [ ]:
def noisy_f(x):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
out = select_exercise(solutions.exercise_5_noisy_f, noisy_f)(torch.tensor([-4., 0., 9.]))
assert torch.allclose(out, torch.tensor([-2., 0., 3.]))
print('exercise 5 OK')

## 6. C51 categorical projection (three stars)
Given a discrete return distribution `p` on a fixed support `support`, compute the projected target distribution $\Phi \mathcal{T} Z$ for one Bellman backup. This is the most subtle algorithm in this curriculum's first half - take your time.

> **Hint**: each transformed atom $\tilde z_j = \text{clip}(r + \gamma z_j, V_{min}, V_{max})$ scatters its mass to the two neighbouring grid atoms with linear-interpolation weights $u - b$ and $b - l$ where $b = (\tilde z - V_{min})/\Delta z$.

In [ ]:
def c51_project(p, support, reward, done, gamma):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
p = torch.tensor([[0.2, 0.3, 0.5]])
support = torch.tensor([-1., 0., 1.])
out = select_exercise(solutions.exercise_6_categorical_project, c51_project)(p, support, torch.tensor([0.1]), torch.tensor([0.0]), 0.9)
assert abs(out.sum().item() - 1.0) < 1e-5
print('exercise 6 OK')
# Atoms map to (-.8,.1,1): split their mass linearly across neighboring bins.
torch.testing.assert_close(out, torch.tensor([[.16,.31,.53]]), atol=1e-6, rtol=0)


## 7. n-step return (two stars)
$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + (1 - d_{t+n}) \gamma^n Q(s_{t+n}, a_{t+n})$.

In [ ]:
def nstep_return(rewards, q_bootstrap, gamma, n, done_last):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
assert select_exercise(solutions.exercise_7_nstep_return, nstep_return)([1, 1, 1], 10.0, 1.0, 2, False) == 12.0
assert select_exercise(solutions.exercise_7_nstep_return, nstep_return)([1, 1, 1], 10.0, 1.0, 2, True) == 2.0
print('exercise 7 OK')

## 8. BBF shrink-and-perturb (two stars)
$\theta \leftarrow \alpha \theta + (1 - \alpha)\theta_{init}$ with $\alpha \approx 0.5$. Reset Adam moments too in practice. This is a simplified reset exercise. The published BBF recipe also includes architecture, replay, auxiliary prediction, and schedule choices; this exercise is not a full reproduction.

In [ ]:
def shrink_perturb(theta, theta_init, alpha):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
th = torch.ones(3); th0 = torch.zeros(3)
assert torch.allclose(select_exercise(solutions.exercise_8_shrink_perturb, shrink_perturb)(th, th0, 1.0), th)
assert torch.allclose(select_exercise(solutions.exercise_8_shrink_perturb, shrink_perturb)(th, th0, 0.0), th0)
print('exercise 8 OK')

## 9. Smoke ablation (three stars, meaningful objective)
**Matching angle.** Run Rainbow-lite vs. no-PER vs. no-Noisy on LunarLander. Compare evaluation returns under matched budgets and seeds. PER reweights replay rows toward a uniform-buffer objective; Week 11 instead reweights logged actions toward a target policy. The change-of-measure idea is shared, but a PER-corrected loss does not itself estimate policy uplift from logged data.

In [ ]:
res = solutions.exercise_9_smoke_ablate(n_steps=1500)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for (name, log), color in zip(res.items(), [RL_PALETTE['blue'], RL_PALETTE['orange'], RL_PALETTE['teal']]):
    rs = np.asarray(log.episode_returns)
    ax.plot(np.arange(len(rs)), rs, label=name, color=color, alpha=0.8)
ax.set_xlabel('episode'); ax.set_ylabel('return'); ax.legend(); ax.set_title('Smoke ablation')
fig.tight_layout(); plt.show()
print({k: v.meta['mean_last_10'] for k, v in res.items()})

## Reflection
1. Which of the Rainbow components addresses *exploration*? Which addresses *value-estimate bias*? Which addresses *sample efficiency*?
2. How do mean- and max-subtraction change the interpretation of V and the gradients, and what experiment would compare them?
3. In what scenario could PER *hurt* rather than help? (Hint: think about high-noise rewards.)
4. BBF resets are a heavy hammer - propose a milder intervention that still combats plasticity loss.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>1. Double DQN target (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>2. Dueling head (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>3. PER importance-sampling weight (two stars)</summary>

> **Hint**: $\beta=1$ gives exact correction before normalisation. A full-buffer maximum rescales the expectation; a minibatch maximum can add bias. common schedule is $\beta: 0.4 \to 1$ linearly over training.

</details>

<details>
<summary>4. PER priority update (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>5. NoisyNet $f$ function (one star)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>6. C51 categorical projection (three stars)</summary>

> **Hint**: each transformed atom $\tilde z_j = \text{clip}(r + \gamma z_j, V_{min}, V_{max})$ scatters its mass to the two neighbouring grid atoms with linear-interpolation weights $u - b$ and $b - l$ where $b = (\tilde z - V_{min})/\Delta z$.

</details>

<details>
<summary>7. n-step return (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>8. BBF shrink-and-perturb (two stars)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>9. Smoke ablation (three stars, meaningful objective)</summary>

(Hint: think about high-noise rewards.)
4. BBF resets are a heavy hammer - propose a milder intervention that still combats plasticity loss.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — A return distribution contains more than its mean

**Goal.** Compare policies with the same return mean but different return tails. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** Two return distributions have support (-2,0,2). One always returns 0; the
other returns -2 or +2 with equal probability. Predict their means, variances,
and lower-tail CVaR at mass 0.25. Does fitting a categorical distribution
automatically make the controller risk-sensitive?

**Work a small case by hand.** For the symmetric-risk policy, E[G]=0.5(-2)+0.5(2)=0 and
Var(G)=0.5×4+0.5×4=4. The certain-zero policy has the same mean and variance
0. The worst 25% probability mass of the risky distribution all lies at -2,
so its lower CVaR is -2; the certain policy's is 0. Knowing a mean alone does
not identify a distribution or its tails.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/02-expectation-variance.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import empirical_cvar
support=np.array([-2.,0.,2.]); pmfs={'Certain zero':np.array([0.,1.,0.]),'Symmetric risk':np.array([.5,0.,.5])}
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
for j,(name,p) in enumerate(pmfs.items()):
    mean=p@support; variance=p@((support-mean)**2)
    exact_samples=np.repeat(support,(p*100).astype(int))
    tail=empirical_cvar(exact_samples,.25)
    print(name,{'mean':mean,'variance':variance,'lower_CVaR_25pct':tail})
    axes[0].bar(support+j*.16-.08,p,width=.16,label=name)
    axes[1].scatter(mean,tail,label=name,s=70,marker=['o','s'][j])
assert all(np.isclose(p@support,0) for p in pmfs.values())
axes[0].set(xlabel='Return',ylabel='Probability');axes[0].legend()
axes[1].set(xlabel='Expected return',ylabel='Lower-tail CVaR (25% mass)',xlim=(-.5,.5))
axes[1].legend();fig.suptitle('Equal expected value can conceal different tail outcomes')
fig.tight_layout();plt.show()


**Interpret and transfer.** A distributional critic models return variability. Choosing actions by the
distribution's mean still optimizes expected return. Tail-aware control needs
an explicitly stated criterion and its own estimation assumptions.

**Transfer.** Add a rare return -20 with probability 0.01 and compensate the
positive tail to keep mean zero. Recalculate CVaR at masses .25 and .01 and
explain how many independent episodes would reveal that tail reliably.


**Read your result.** The right-panel points share the same horizontal coordinate but have different
vertical coordinates. A distributional critic contains information that a
mean-greedy action rule may choose not to use.

<details><summary>Worked feedback for the changed assumption</summary>

One zero-mean rare-loss example puts probability 0.01 at -20 and the remaining
0.99 at 20/99≈0.2020. Its lower 1% CVaR is -20. Its lower 25% tail contains
the loss plus 0.24 positive mass, giving
(-0.2+0.24×20/99)/0.25≈-0.6061. At least 299 independent episodes are needed
for a 95% chance of even one 1%-probability loss, since 0.99^299<0.05.
Seeing one loss still does not estimate its tail precisely.

</details>
